# Day 13: Testing and Data Engineering Intro
**PyTest, React testing, the data lifecycle, ETL vs ELT and data ingestion**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q pytest fastapi httpx pandas
!node --version

# ===== Option B: VS Code / local (run in your activated venv terminal) =====
# pip install pytest fastapi httpx pandas
# node --version
import os
os.makedirs("day13", exist_ok=True)
os.chdir("day13")
print("Working in", os.getcwd())

## 2. PyTest basics

In [ ]:
%%writefile pricing.py
def with_gst(amount, rate=18):
    if amount < 0:
        raise ValueError("amount must not be negative")
    return round(amount * (1 + rate / 100), 2)

def grade(marks):
    if not 0 <= marks <= 100:
        raise ValueError("marks must be 0 to 100")
    if marks >= 85:
        return "A"
    if marks >= 70:
        return "B"
    if marks >= 50:
        return "C"
    return "Needs improvement"

In [ ]:
%%writefile test_pricing.py
import pytest
from pricing import with_gst, grade

def test_default_rate():
    assert with_gst(100) == 118.0

def test_negative_raises():
    with pytest.raises(ValueError):
        with_gst(-1)

@pytest.mark.parametrize("amount, rate, expected", [(100, 18, 118.0), (100, 5, 105.0), (0, 18, 0.0)])
def test_rates(amount, rate, expected):
    assert with_gst(amount, rate) == expected

@pytest.mark.parametrize("marks, expected", [(0, "Needs improvement"), (49, "Needs improvement"), (50, "C"),
                                             (69, "C"), (70, "B"), (84, "B"), (85, "A"), (100, "A")])
def test_grade_boundaries(marks, expected):
    assert grade(marks) == expected

def test_grade_out_of_range():
    with pytest.raises(ValueError):
        grade(101)

In [ ]:
# ===== Option A: Google Colab (active) =====
!python -m pytest -q test_pricing.py

# ===== Option B: VS Code =====
# Terminal: pytest -q    (or use the Testing panel, the flask icon in the sidebar)

## 3. Testing a FastAPI app with fixtures

In [ ]:
%%writefile tasks_api.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

app = FastAPI()
tasks: dict[int, dict] = {}

class TaskIn(BaseModel):
    title: str = Field(min_length=1)

@app.get("/tasks")
def list_tasks():
    return list(tasks.values())

@app.post("/tasks", status_code=201)
def create(t: TaskIn):
    tid = max(tasks, default=0) + 1
    tasks[tid] = {"id": tid, "title": t.title, "done": False}
    return tasks[tid]

@app.get("/tasks/{tid}")
def get(tid: int):
    if tid not in tasks:
        raise HTTPException(404, "Task not found")
    return tasks[tid]

@app.delete("/tasks/{tid}", status_code=204)
def delete(tid: int):
    if tasks.pop(tid, None) is None:
        raise HTTPException(404, "Task not found")

In [ ]:
%%writefile test_tasks_api.py
import pytest
from fastapi.testclient import TestClient
import tasks_api

@pytest.fixture
def client():
    tasks_api.tasks.clear()          # every test starts with an empty store
    return TestClient(tasks_api.app)

def test_starts_empty(client):
    assert client.get("/tasks").json() == []

def test_create_returns_201(client):
    r = client.post("/tasks", json={"title": "Write tests"})
    assert r.status_code == 201
    assert r.json() == {"id": 1, "title": "Write tests", "done": False}

def test_empty_title_is_422(client):
    assert client.post("/tasks", json={"title": ""}).status_code == 422

def test_missing_is_404(client):
    assert client.get("/tasks/999").status_code == 404

def test_delete_twice(client):
    client.post("/tasks", json={"title": "x"})
    assert client.delete("/tasks/1").status_code == 204
    assert client.delete("/tasks/1").status_code == 404

In [ ]:
# ===== Option A: Google Colab (active) =====
!python -m pytest -q test_tasks_api.py

# ===== Option B: VS Code =====
# Terminal: pytest -q test_tasks_api.py

### TODO 1: Add a test that creating two tasks gives ids 1 and 2, and a test that GET /tasks returns both.

In [ ]:
# TODO: write your code here

## 4. JavaScript unit tests with Node's built-in runner
No installation needed. In a React project you would use Vitest and React Testing Library (see the deck); the ideas are the same.

In [ ]:
%%writefile cart.mjs
export function formatPrice(n) {
  return "Rs " + n.toFixed(2);
}

export function cartTotal(items) {
  return items.reduce((sum, i) => sum + i.price * i.qty, 0);
}

In [ ]:
%%writefile cart.test.mjs
import test from "node:test";
import assert from "node:assert/strict";
import { formatPrice, cartTotal } from "./cart.mjs";

test("formats with two decimals", () => {
  assert.equal(formatPrice(5), "Rs 5.00");
});

test("empty cart totals zero", () => {
  assert.equal(cartTotal([]), 0);
});

test("multiplies price by quantity", () => {
  assert.equal(cartTotal([{ price: 100, qty: 2 }, { price: 50, qty: 1 }]), 250);
});

In [ ]:
# ===== Option A: Google Colab (active) =====
!node --test cart.test.mjs

# ===== Option B: VS Code =====
# Terminal: node --test cart.test.mjs

## 5. A tiny ETL pipeline
First we create a deliberately messy CSV, the kind real systems produce.

In [ ]:
%%writefile orders.csv
order_id,customer,amount,order_date,city
1001, Asha ,250.50,2026-11-01,Bengaluru
1002,Kenji,1200,2026-11-01,Tokyo
1002,Kenji,1200,2026-11-01,Tokyo
1003,Mei,,2026-11-02,Osaka
1004,Ravi,abc,2026-11-02,Pune
1005,Priya,560,02/11/2026,Chennai
1006, Hana,90.25,2026-11-03,Kyoto
1007,Arjun,3100,2026-11-03,Bengaluru
1007,Arjun,3100,2026-11-03,Bengaluru
1008,Yuki,45,2026-11-04,Tokyo
1009,Sara,780,2026-11-04,Pune
1010,Ken,150,2026-11-05,Osaka

In [ ]:
import pandas as pd, sqlite3

def extract(path):
    return pd.read_csv(path)

def transform(df):
    df = df.drop_duplicates("order_id").copy()
    df["customer"] = df["customer"].str.strip()
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    df["order_date"] = pd.to_datetime(df["order_date"], format="mixed", dayfirst=True).dt.date.astype(str)
    return df.dropna(subset=["amount"])

def load(df, conn, table="orders"):
    df.to_sql(table, conn, if_exists="replace", index=False)
    return len(df)

raw = extract("orders.csv")
clean = transform(raw)
conn = sqlite3.connect("shop.db")
print("Raw rows:", len(raw), "-> Clean rows:", len(clean))
print("Loaded", load(clean, conn), "rows")
print(pd.read_sql("SELECT city, COUNT(*) n, SUM(amount) total FROM orders GROUP BY city ORDER BY total DESC", conn))

## 6. The ELT version: load raw, transform with SQL

In [ ]:
raw.to_sql("raw_orders", conn, if_exists="replace", index=False)
conn.executescript('''
DROP TABLE IF EXISTS orders_clean;
CREATE TABLE orders_clean AS
SELECT DISTINCT order_id, TRIM(customer) AS customer, CAST(amount AS REAL) AS amount, order_date, city
FROM raw_orders
WHERE amount IS NOT NULL AND CAST(amount AS REAL) > 0;
''')
print("raw_orders:", pd.read_sql("SELECT COUNT(*) n FROM raw_orders", conn).n[0])
print("orders_clean:", pd.read_sql("SELECT COUNT(*) n FROM orders_clean", conn).n[0])

## 7. Idempotent incremental load

In [ ]:
def load_incremental(new_df, conn, table="orders"):
    existing = set(pd.read_sql(f"SELECT order_id FROM {table}", conn)["order_id"])
    fresh = new_df[~new_df["order_id"].isin(existing)]
    fresh.to_sql(table, conn, if_exists="append", index=False)
    return len(fresh)

new_batch = pd.DataFrame([
    {"order_id": 1010, "customer": "Ken", "amount": 150.0, "order_date": "2026-11-05", "city": "Osaka"},
    {"order_id": 1011, "customer": "Zara", "amount": 990.0, "order_date": "2026-11-06", "city": "Delhi"},
])
print("Run 1 inserted:", load_incremental(new_batch, conn))
print("Run 2 inserted:", load_incremental(new_batch, conn))
print("Total rows:", pd.read_sql("SELECT COUNT(*) n FROM orders", conn).n[0])

### TODO 2: Write PyTest data quality tests (in a `test_quality.py` file) that check there are no duplicate order ids and no negative amounts in the `orders` table.

In [ ]:
# TODO: write your code here

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)